In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = r"C:\Users\PC\Downloads\karthika journal.pdf"

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print("Pages loaded:", len(pages))

C:\Users\PC\AppData\Local\Temp\ipykernel_13284\3840211371.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages loaded: 5


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print("Number of chunks:", len(chunks))

Number of chunks: 48


In [5]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv(r"C:\Users\PC\Desktop\100 DAYS OF AI\WEEK 1\DAY 1\.env")

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("GEMINI_API_KEY")
)

chunk_texts = [chunk.page_content for chunk in chunks]

chunk_vectors = embeddings.embed_documents(chunk_texts)

print("Number of vectors:", len(chunk_vectors))
print("Vector dimensions:", len(chunk_vectors[0]))

Number of vectors: 48
Vector dimensions: 3072


In [6]:
import chromadb

client = chromadb.PersistentClient(path="../DAY 18/chroma_db")

collection = client.get_collection(name="vehicle_damage_docs")

print("Chroma collection loaded!")
print("Number of documents:", collection.count())

Chroma collection loaded!
Number of documents: 48


In [7]:
query = "What are the three severity levels?"

query_vector = embeddings.embed_query(query)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3
)

context = "\n\n".join(results["documents"][0])

print("Retrieved Context:\n")
print(context)

Retrieved Context:

moderate, indicating more significant damage covering a larger 
surface area or affecting panel alignment; and severe, 
representing extensive structural damage requiring major repair 
intervention. The severity assignment assists users and 
insurance assessors in making informed decisions regarding 
repair prioritization and claim processing.   
E.  Database and Output Module  
Detection results including bounding box coordinates, 
damage categories, confidence scores , and severity

separation or missing vehicle parts. 
D.  Severity Classification Module  
Following damage detection, the severity classification 
module analyses the spatial extent, depth characteristics, and 
distribution of each identified damage region to assign a 
severity rating. Damage instances are categorized into three 
levels: minor, referring to superficial surface marks with 
limited extent th at do not affect vehicle structural integrity;

characteristics. The severity estimation module

In [8]:
prompt = f"""
Answer the question using only the context provided below.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say:
"I could not find the answer in the provided document."

Answer clearly and concisely.
"""

response = llm.invoke(prompt)

answer = response.content[0]["text"]

print("Question:", query)
print("\nGenerated Answer:", answer)

C:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: What are the three severity levels?

Generated Answer: Based on the provided context, the three severity levels are minor, moderate, and severe.


In [9]:
print("Question:", query)
print("\nGenerated Answer:", answer)

Question: What are the three severity levels?

Generated Answer: Based on the provided context, the three severity levels are minor, moderate, and severe.


In [10]:
evaluation_prompt = f"""
Evaluate how relevant and correct the answer is for the question,
using the provided context.

Question:
{query}

Context:
{context}

Answer:
{answer}

Score the answer from 1 to 5:

5 = Completely correct and relevant
4 = Mostly correct and relevant
3 = Partially correct
2 = Mostly incorrect or irrelevant
1 = Completely incorrect or irrelevant

Return only:
Score: <number>/5
Reason: <one short sentence>
"""

evaluation = llm.invoke(evaluation_prompt)

print(evaluation.content[0]["text"])

C:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Score: 5/5
Reason: The answer directly and accurately identifies the three severity levels mentioned in the context.


In [11]:
bad_answer = "The vehicle damage categories are cars, bikes, trucks, and buses."

evaluation_prompt = f"""
Evaluate how relevant and correct the answer is for the question,
using the provided context.

Question:
{query}

Context:
{context}

Answer:
{bad_answer}

Score the answer from 1 to 5:

5 = Completely correct and relevant
4 = Mostly correct and relevant
3 = Partially correct
2 = Mostly incorrect or irrelevant
1 = Completely incorrect or irrelevant

Return only:
Score: <number>/5
Reason: <one short sentence>
"""

evaluation = llm.invoke(evaluation_prompt)

print(evaluation.content[0]["text"])

C:\Users\PC\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Score: 1/5
Reason: The answer incorrectly lists vehicle types instead of identifying the three severity levels (minor, moderate, and severe) specified in the context.


# Conclusion

Today I learned how to evaluate the quality of a RAG-generated answer using a simple LLM-based evaluation approach.

I built a pipeline that retrieves relevant context from my PDF, generates an answer using Gemini, and then asks Gemini to evaluate the answer on a scale of 1 to 5.

I tested both a correct answer and an incorrect answer. The correct answer received a score of 5/5, while the incorrect answer received 1/5.

### Evaluation Flow

Question → Retrieve Context → Generate Answer → Evaluate Answer → Score

### Key Takeaways

- RAG evaluation helps measure the quality of generated answers.
- An evaluator can compare an answer against the retrieved context.
- A simple 1–5 relevance score can be used to evaluate answers.
- Correct answers should receive higher scores than incorrect answers.
- Evaluation is important for improving the reliability of RAG systems.

**Day 19 Complete! 🚀**

In [12]:
readme = """# Day 19 - RAG Evaluation

## Objective

Evaluate the relevance and correctness of answers generated by a RAG pipeline.

## What I Built

I created a simple LLM-based evaluation system that:

1. Retrieves relevant context from a PDF using Chroma.
2. Generates an answer using Gemini.
3. Evaluates the generated answer using Gemini.
4. Assigns a relevance score from 1 to 5.

## Evaluation Flow

Question → Retrieve Context → Generate Answer → Evaluate Answer → Score

## Test Results

| Answer Type | Score |
|---|---:|
| Correct answer | 5/5 |
| Incorrect answer | 1/5 |

## Technologies Used

- Python
- LangChain
- Gemini
- Chroma
- PyPDF

## Key Takeaways

- RAG evaluation helps measure answer quality.
- An LLM can be used as a simple evaluator.
- The evaluator compares the answer with the question and retrieved context.
- A 1–5 score provides a simple way to measure relevance.
- Testing both correct and incorrect answers helps verify that the evaluator works.

## Conclusion

Today I learned how to evaluate RAG-generated answers using a simple LLM-based evaluation approach. The system successfully gave a high score to a correct answer and a low score to an incorrect answer.

**Day 19 Complete! 🚀**
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("README.md created successfully!")

README.md created successfully!
